# Build Dataset: Geopotential Height 850 hPa Correlation Cache

Build the 850 hPa geopotential height DJF correlation cache against Niño3.4 anomaly.

This notebook keeps the exact DJF correlation logic from `correlation_mc.ipynb`, but strips all plotting. The output is a NetCDF cache that can be reused later when plot styling or annotations change.

# 1. Import Library

In [ ]:

from pathlib import Path
import numpy as np
import pandas as pd
import xarray as xr
from scipy.stats import norm, t as student_t

# Keep the same study window and split years as correlation_mc.ipynb.
FULL_YEARS = np.arange(1981, 2026)
PAST_YEARS = np.arange(1981, 2007)
RECENT_YEARS = np.arange(2007, 2026)
ANALYSIS_START = '1980-12-01'
ANALYSIS_END = '2025-02-28'
MC_LAT = slice(32, -32)
MC_LON = slice(58, 272)
GRAVITY = 9.80665
EARTH_RADIUS = 6371000.0

RAINFALL_PATH = Path('/Users/rizzie/ClimateData/mswep-monthly/mswep_monthly_combined.nc').resolve()
WIND_PATH = Path('/Users/rizzie/ClimateData/era5-monthly/uv_850_global_1979-2025.nc').resolve()
MSLP_PATH = Path('/Users/rizzie/ClimateData/era5-monthly/mslp_global_1979-2025.nc').resolve()
GEO850_PATH = Path('/Users/rizzie/ClimateData/era5-monthly/geopotensial_850_global_1979-2025.nc').resolve()
MFC_PATH = Path('/Users/rizzie/ClimateData/era5-monthly/viwve_viwvn_global_1979-2025.nc').resolve()
SVP_PATH = Path('/Users/rizzie/ClimateData/era5-monthly/psi_chi_850_global_1979-2025.nc').resolve()
NINO34_PATH = Path('/Users/rizzie/ClimateData/index-monthly/nino34.anom.csv').resolve()
NINO34_COLUMN = '   Nino Anom 3.4 Index  using ersstv5 from CPC  missing value -99.99 https://psl.noaa.gov/data/timeseries/month/'


def standardize_obj(obj):
    rename_map = {}
    if 'valid_time' in obj.coords or 'valid_time' in obj.dims:
        rename_map['valid_time'] = 'time'
    if 'latitude' in obj.coords or 'latitude' in obj.dims:
        rename_map['latitude'] = 'lat'
    if 'longitude' in obj.coords or 'longitude' in obj.dims:
        rename_map['longitude'] = 'lon'
    if rename_map:
        obj = obj.rename(rename_map)

    if 'lon' in obj.coords:
        obj = obj.assign_coords(lon=(obj.lon % 360)).sortby('lon')

    if 'lat' in obj.coords:
        lat_values = np.asarray(obj['lat'].values)
        if lat_values.size > 1 and lat_values[0] < lat_values[-1]:
            obj = obj.sortby('lat', ascending=False)

    return obj


def select_850_hpa_if_present(obj):
    for level_name in ('pressure_level', 'level', 'lev'):
        if level_name not in obj.coords and level_name not in obj.dims:
            continue

        level_values = np.atleast_1d(np.asarray(obj[level_name].values))
        for target in (850, 850.0, 85000, 85000.0):
            if target in level_values:
                return obj.sel({level_name: target})

        raise ValueError(
            f"850 hPa level not found in '{level_name}'. Available values include: "
            f"{level_values[:10]}"
        )

    return obj


def match_variable_name(ds, target):
    target_key = target.replace('_', '').lower()
    matches = [
        name
        for name in ds.data_vars
        if name.lower() == target.lower()
        or name.lower().replace('_', '') == target_key
    ]
    if not matches:
        matches = [
            name
            for name in ds.data_vars
            if target_key in name.lower().replace('_', '')
        ]
    if not matches:
        raise KeyError(f'Variable for {target} not found in the dataset')
    return matches[0]


def crop_analysis_domain(obj):
    if 'lat' in obj.coords or 'lat' in obj.dims:
        obj = obj.sel(lat=MC_LAT)
    if 'lon' in obj.coords or 'lon' in obj.dims:
        obj = obj.sel(lon=MC_LON)
    return obj


def build_djf_seasonal_means(da, start=ANALYSIS_START, end=ANALYSIS_END):
    da = da.sel(time=slice(start, end))
    da = crop_analysis_domain(da)
    month_mask = da.time.dt.month.isin((12, 1, 2))
    djf_year = xr.where(da.time.dt.month == 12, da.time.dt.year + 1, da.time.dt.year)
    da = da.sel(time=month_mask).assign_coords(
        djf_year=('time', djf_year.sel(time=month_mask).data)
    )
    da = da.sel(time=da.djf_year.isin(FULL_YEARS))
    clim = da
    past = clim.sel(time=clim.djf_year.isin(PAST_YEARS))
    recent = clim.sel(time=clim.djf_year.isin(RECENT_YEARS))
    return (
        clim.groupby('djf_year').mean('time'),
        past.groupby('djf_year').mean('time'),
        recent.groupby('djf_year').mean('time'),
    )


def build_nino34_djf_series(df):
    df = df.set_index('Date').loc['1980-12-01':'2025-02-28']
    df = df[df.index.month.isin((12, 1, 2))].copy()
    df['djf_year'] = df.index.year + (df.index.month == 12).astype('int8')
    df = df[df['djf_year'].isin(FULL_YEARS)].copy()

    clim_series = df.groupby('djf_year')[NINO34_COLUMN].mean()
    past_series = df[df['djf_year'].isin(PAST_YEARS)].groupby('djf_year')[NINO34_COLUMN].mean()
    recent_series = df[df['djf_year'].isin(RECENT_YEARS)].groupby('djf_year')[NINO34_COLUMN].mean()

    clim = xr.DataArray(
        clim_series.to_numpy(),
        coords={'djf_year': clim_series.index.to_numpy()},
        dims='djf_year',
        name='nino34',
    )
    past = xr.DataArray(
        past_series.to_numpy(),
        coords={'djf_year': past_series.index.to_numpy()},
        dims='djf_year',
        name='nino34',
    )
    recent = xr.DataArray(
        recent_series.to_numpy(),
        coords={'djf_year': recent_series.index.to_numpy()},
        dims='djf_year',
        name='nino34',
    )
    return clim, past, recent


def safe_corr(corr):
    return xr.where(np.abs(corr) >= 0.999999, np.sign(corr) * 0.999999, corr)


def corr_and_pvalues(field_clim, field_past, field_recent, nino_clim, nino_past, nino_recent):
    field_clim, nino_clim = xr.align(field_clim, nino_clim, join='inner')
    field_past, nino_past = xr.align(field_past, nino_past, join='inner')
    field_recent, nino_recent = xr.align(field_recent, nino_recent, join='inner')

    corr_clim = xr.corr(field_clim, nino_clim, dim='djf_year').compute()
    corr_past = xr.corr(field_past, nino_past, dim='djf_year').compute()
    corr_recent = xr.corr(field_recent, nino_recent, dim='djf_year').compute()
    corr_recent_minus_past = corr_recent - corr_past

    n_full = int(field_clim.sizes['djf_year'])
    n_past = int(field_past.sizes['djf_year'])
    n_recent = int(field_recent.sizes['djf_year'])

    corr_clim_safe = safe_corr(corr_clim)
    corr_past_safe = safe_corr(corr_past)
    corr_recent_safe = safe_corr(corr_recent)

    corr_clim_t = corr_clim_safe * np.sqrt((n_full - 2) / (1 - corr_clim_safe**2))
    corr_past_t = corr_past_safe * np.sqrt((n_past - 2) / (1 - corr_past_safe**2))
    corr_recent_t = corr_recent_safe * np.sqrt((n_recent - 2) / (1 - corr_recent_safe**2))

    corr_clim_p = xr.apply_ufunc(
        lambda x: 2 * student_t.sf(np.abs(x), df=n_full - 2),
        corr_clim_t,
        vectorize=True,
        dask='allowed',
        output_dtypes=[float],
    ).compute()
    corr_past_p = xr.apply_ufunc(
        lambda x: 2 * student_t.sf(np.abs(x), df=n_past - 2),
        corr_past_t,
        vectorize=True,
        dask='allowed',
        output_dtypes=[float],
    ).compute()
    corr_recent_p = xr.apply_ufunc(
        lambda x: 2 * student_t.sf(np.abs(x), df=n_recent - 2),
        corr_recent_t,
        vectorize=True,
        dask='allowed',
        output_dtypes=[float],
    ).compute()

    z_stat = (np.arctanh(corr_recent_safe) - np.arctanh(corr_past_safe)) / np.sqrt(
        1 / (n_recent - 3) + 1 / (n_past - 3)
    )
    corr_recent_minus_past_p = xr.apply_ufunc(
        lambda x: 2 * norm.sf(np.abs(x)),
        z_stat,
        vectorize=True,
        dask='allowed',
        output_dtypes=[float],
    ).compute()

    return {
        'corr_clim': corr_clim,
        'corr_past': corr_past,
        'corr_recent': corr_recent,
        'corr_recent_minus_past': corr_recent_minus_past,
        'p_clim': corr_clim_p,
        'p_past': corr_past_p,
        'p_recent': corr_recent_p,
        'p_recent_minus_past': corr_recent_minus_past_p,
    }


def cast_float_and_mask_vars(ds):
    ds = ds.copy()
    for name in list(ds.data_vars):
        dtype = ds[name].dtype
        if np.issubdtype(dtype, np.floating):
            ds[name] = ds[name].astype('float32')
        elif np.issubdtype(dtype, np.bool_):
            ds[name] = ds[name].astype('int8')
    return ds


def build_encoding(ds):
    return {name: {'zlib': True, 'complevel': 4} for name in ds.data_vars}


def write_dataset(ds, path, label):
    ds = cast_float_and_mask_vars(ds)
    print(f'Writing {label}: {path}')
    ds.to_netcdf(path, encoding=build_encoding(ds))


# 2. Open Data

In [ ]:
gh850_path = GEO850_PATH
gh850_chunks = {'valid_time': 12}

ds_gh850 = xr.open_dataset(gh850_path, chunks=gh850_chunks)['z']
ds_gh850 = ds_gh850.sel(pressure_level=850)
ds_gh850 = standardize_obj(ds_gh850)
gh850_raw = (ds_gh850 / GRAVITY).rename('gh850')
gh850_raw.attrs['units'] = 'm'
gh850_raw.attrs['standard_name'] = 'geopotential_height'
gh850_raw.attrs['long_name'] = 'Geopotential height at 850 hPa'

# 3. Correlation & Significance

In [ ]:
df_nino34 = pd.read_csv(
    NINO34_PATH,
    usecols=['Date', NINO34_COLUMN],
    parse_dates=['Date'],
)
nino34_clim, nino34_past, nino34_recent = build_nino34_djf_series(df_nino34)

gh850_clim, gh850_past, gh850_recent = build_djf_seasonal_means(gh850_raw)
gh850_stats = corr_and_pvalues(
    gh850_clim, gh850_past, gh850_recent,
    nino34_clim, nino34_past, nino34_recent,
)

# 4. Save Cache

In [ ]:
cache_path = Path('/Users/rizzie/TugasAkhir/data_processing/data/intermediate/divided_correlation/gh850_corr_cache_1981_2025.nc').resolve()
cache_path.parent.mkdir(parents=True, exist_ok=True)

cache_ds = xr.Dataset(
    {
        'gh850_corr_clim': gh850_stats['corr_clim'],
        'gh850_corr_past': gh850_stats['corr_past'],
        'gh850_corr_recent': gh850_stats['corr_recent'],
        'gh850_corr_recent_minus_past': gh850_stats['corr_recent_minus_past'],
        'gh850_corr_clim_p': gh850_stats['p_clim'],
        'gh850_corr_past_p': gh850_stats['p_past'],
        'gh850_corr_recent_p': gh850_stats['p_recent'],
        'gh850_corr_recent_minus_past_p': gh850_stats['p_recent_minus_past'],
        'gh850_corr_clim_sig': (gh850_stats['p_clim'] < 0.05),
        'gh850_corr_past_sig': (gh850_stats['p_past'] < 0.05),
        'gh850_corr_recent_sig': (gh850_stats['p_recent'] < 0.05),
        'gh850_corr_recent_minus_past_sig': (gh850_stats['p_recent_minus_past'] < 0.05),
    }
)
cache_ds.attrs.update(
    {
        'title': 'Geopotential height 850 hPa correlation cache',
        'period': '1981-2025',
        'split_p1': '1981-2006',
        'split_p2': '2007-2025',
        'note': 'Geopotential height uses raw monthly values converted from z / g; Niño3.4 stays on the anomaly CSV.',
    }
)
write_dataset(cache_ds, cache_path, 'gh850 cache')